# 04 (Optional) Extra data: เพิ่ม CoFly-WeedDB เป็น class ที่สอง แล้วเทรนเทียบ

> **Optional: ไม่ต้องรันไฟล์นี้** เป็นการทดลองที่จบแล้ว ผลคือไม่ช่วย โมเดลที่เลือกใช้ไม่ได้ใช้ข้อมูลนี้
> ชุดข้อมูลรวม `data/weedyrice_cofly_seg` ถูกลบแล้วเพื่อประหยัดพื้นที่ ถ้ารันข้อ 3 จะสร้างใหม่จาก `data/raw/cofly`

การทดลอง: เพิ่มภาพโดรนแปลงฝ้าย (CoFly-WeedDB) เข้า **train เท่านั้น** เป็น class ใหม่ `other_weed`
แล้วดูว่าช่วยให้จับ `weedy_rice` บน val ดีขึ้นไหม เทียบกับ `yolo26n-seg` รอบแรก

- kernel **`Python (rwd)`**, ต้องใช้ GPU ตอนเทรน, ต้องมี `data/weedyrice_seg` จาก `01_data_setup.ipynb` ก่อน
- CoFly-WeedDB: Krestenitis et al. 2022, Zenodo `10.5281/zenodo.6697343`, **CC BY 4.0** ดู `DATASET.md` **ไม่ใช่ข้อมูลของเราเอง**
- วัชพืชใน CoFly **ไม่ใช่ข้าววัชพืช** จึงแยกเป็น class `other_weed` ไม่รวมกับ `weedy_rice`
- val และ test เหมือนเดิมทุกภาพ และวัดผลเฉพาะ class `weedy_rice` จึงเทียบกับรอบก่อนได้ตรง ๆ
- วัดบน **val เท่านั้น** ไฟล์นี้ไม่แตะ test
- รันซ้ำได้ทุก cell: ถ้าโหลด สร้างชุดข้อมูล หรือเทรนไปแล้ว จะข้ามเอง

## 1. Setup

In [ ]:
import os, sys, json, csv, subprocess
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
PY = sys.executable
if ".venv" not in PY:
    print("WARNING: kernel is not 'Python (rwd)'. Run scripts/setup_env.sh, then Kernel -> Change Kernel.")

from rwd.train import load_train_config, run_state
DATA_CONFIG = ROOT / "configs/cofly.yaml"            # วิธีรวมชุดข้อมูล
CONFIG = ROOT / "configs/train_cofly.yaml"           # การเทรนรอบนี้
BASE_CONFIG = ROOT / "configs/train.yaml"            # รอบแรกที่ใช้เทียบ (WeedyRice อย่างเดียว)
RAW = ROOT / "data/raw/cofly/CoFly-WeedDB"
OUT = ROOT / "data/weedyrice_cofly_seg"
cfg, base_cfg = load_train_config(CONFIG), load_train_config(BASE_CONFIG)
RUN, BASE_RUN = Path(cfg["project"]) / cfg["name"], Path(base_cfg["project"]) / base_cfg["name"]
LOG = ROOT / "train.log"
print("run:", RUN.name, "| state:", run_state(RUN))
print("เทียบกับ:", BASE_RUN.name, "| state:", run_state(BASE_RUN))

## 2. โหลด CoFly-WeedDB (1.2 GB จาก Zenodo)

แตกเฉพาะ 201 ภาพที่มี mask, โฟลเดอร์ `labels_1d` และไฟล์ split ไว้ที่ `data/raw/cofly/` (ไม่เข้า git)
ถ้ามีอยู่แล้วจะข้าม

In [ ]:
import urllib.request, zipfile

URL = "https://zenodo.org/api/records/6697343/files/CoFly-WeedDB.zip/content"
ZIP = ROOT / "data/raw/cofly_dl/CoFly-WeedDB.zip"
if (RAW / "labels_1d").exists():
    print("มีอยู่แล้ว ข้ามการโหลด")
else:
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP.exists():
        print("กำลังโหลด 1.2 GB ...")
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        labelled = {n.split("/")[-1] for n in z.namelist() if "/labels_1d/" in n}
        for n in z.namelist():
            folder = n.split("/")[1] if "/" in n else ""
            if n.endswith("/") or folder in ("overlay", "labels") or (folder == "images" and n.split("/")[-1] not in labelled):
                continue
            z.extract(n, RAW.parent)
print("images:", len(list((RAW / "images").glob("*.png"))), "| masks:", len(list((RAW / "labels_1d").glob("*.png"))))

## 3. สร้างชุดข้อมูลรวม

`rwd.add_extra` คัดลอก WeedyRice ทั้งชุดแบบไม่แก้ แล้วเพิ่ม CoFly 201 ภาพเข้า train เป็น class 1 `other_weed`
คำเตือนว่า `other_weed` ไม่มีใน val และ test เป็นเรื่องตั้งใจ

In [ ]:
OVERWRITE = False
if (OUT / "data.yaml").exists() and not OVERWRITE:
    print("สร้างไว้แล้ว:", OUT, "(ตั้ง OVERWRITE = True ถ้าต้องการสร้างใหม่)")
else:
    flag = "--overwrite" if OVERWRITE else ""
    !"{PY}" -m rwd.add_extra --config "{DATA_CONFIG}" {flag}

report = json.loads((OUT / "validation_report.json").read_text())
print("\nตรวจชุดข้อมูล:", "OK" if report["ok"] else "FAILED")
for split, r in report["splits"].items():
    print(f"  {split:<5} images={r['images']:>4}  instances={r['instances']}")
print("  groups:", report["leakage"]["groups_per_split"])
for e in report["errors"]:
    print("  ERROR:", e)

## 4. ดูตัวอย่างภาพ CoFly กับ label

สีม่วง = polygon ของ `other_weed` สังเกตว่าเป็นแปลงฝ้ายบนดินแห้ง และ polygon วาดหยาบกว่าชุด WeedyRice

In [ ]:
import random
import matplotlib.pyplot as plt
from rwd.viz import draw_labels

cofly = sorted((OUT / "images/train").glob("cofly_*.png"))
picked = random.Random(0).sample(cofly, 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, p in zip(axes.flat, picked):
    ax.imshow(draw_labels(p, "segment", num_classes=2)); ax.set_title(p.name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train / Continue

เหมือน `02_train.ipynb`: รันซ้ำได้เสมอ เทรนเป็น process แยก ปิด VS Code ได้ ถ้าเทรนค้างจะต่อจาก checkpoint ถ้าจบแล้วจะข้าม
ดู log สด: `tail -f ~/rice-weed-detection/train.log`

In [ ]:
def training_pids():
    out = subprocess.run(["pgrep", "-f", "rwd.train --config"], capture_output=True, text=True).stdout.split()
    return [p for p in out if p != str(os.getpid())]

if run_state(RUN) == "finished":
    print("เทรนจบแล้ว ข้าม")
elif training_pids():
    print("การเทรนกำลังรันอยู่แล้ว (pid", training_pids()[0] + ") ไม่เริ่มซ้ำ")
else:
    with open(LOG, "a") as f:
        proc = subprocess.Popen([PY, "-m", "rwd.train", "--config", str(CONFIG), "--auto"],
                                stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
    print("started in background, pid", proc.pid, "| log:", LOG)

## 6. Progress

รันซ้ำได้ระหว่างเทรน ค่าในกราฟคือ instance metric บน val (เฉพาะ class ที่มีใน val คือ `weedy_rice`)

In [ ]:
print("state:", run_state(RUN), "| running:", bool(training_pids()))
res = RUN / "results.csv"
if not res.exists():
    print("ยังไม่มี results.csv (ยังไม่จบ epoch แรก)")
else:
    rows = [{k.strip(): float(v) for k, v in r.items()} for r in csv.DictReader(open(res))]
    base_rows = [{k.strip(): float(v) for k, v in r.items()} for r in csv.DictReader(open(BASE_RUN / "results.csv"))]
    print(f"epoch {int(rows[-1]['epoch'])}/{cfg['epochs']}  เวลาที่ใช้ {rows[-1]['time'] / 60:.0f} นาที")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, key in zip(axes, ("metrics/recall(M)", "metrics/mAP50(M)")):
        ax.plot([r["epoch"] for r in base_rows], [r[key] for r in base_rows], label="WeedyRice only")
        ax.plot([r["epoch"] for r in rows], [r[key] for r in rows], label="+ CoFly (other_weed)")
        ax.set_title("val " + key + " (not test)"); ax.set_xlabel("epoch"); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

## 7. วัดบน val และเทียบกับรอบแรก

ตัววัดหลัก: precision / recall / IoU ระดับพิกเซลของ **`weedy_rice` เท่านั้น** ส่วนที่โมเดลทายเป็น `other_weed` ไม่ถูกนับ
val มี 41 ภาพจากวันเดียว ต่างกันเล็กน้อยถือว่าเสมอ **ตัวเลข val ห้ามนำไปเผยแพร่**

In [ ]:
CONFS = [0.05, 0.1, 0.15, 0.25, 0.4, 0.5]
VAL_JSON = RUN / "eval_val.json"
if run_state(RUN) != "finished":
    print("ยังเทรนไม่จบ รอให้จบแล้วรัน cell นี้อีกครั้ง")
else:
    if not VAL_JSON.exists():
        confs = " ".join(str(c) for c in CONFS)
        !"{PY}" -m rwd.evaluate --weights "{RUN / 'weights/best.pt'}" --data "{cfg['data']}" --split val --conf {confs}
    new, old = json.loads(VAL_JSON.read_text()), json.loads((BASE_RUN / "eval_val.json").read_text())
    print("[val: ใช้เลือกโมเดลเท่านั้น ไม่ใช่ผลที่เผยแพร่ได้] class: weedy_rice\n")
    print(f"{'':>6} {'WeedyRice only':^26} | {'+ CoFly (other_weed)':^26}")
    print(f"{'conf':>6} {'precision':>9} {'recall':>7} {'IoU':>7} | {'precision':>9} {'recall':>7} {'IoU':>7}")
    for conf in new["pixel"]:
        a, b = old["pixel"][conf]["all"], new["pixel"][conf]["all"]
        print(f"{conf:>6} {a['precision']:>9.3f} {a['recall']:>7.3f} {a['iou']:>7.3f} | {b['precision']:>9.3f} {b['recall']:>7.3f} {b['iou']:>7.3f}")
    print("\ninstance mask ของ YOLO (precision / recall / mAP50 / mAP50-95):")
    for name, r in (("WeedyRice only", old), ("+ CoFly", new)):
        i = r["instance_default_conf"]
        print(f"  {name:<16}", "  ".join(f"{i[k]:.3f}" for k in ("metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(M)", "metrics/mAP50-95(M)")))
    best = lambda r: max((v["all"]["iou"], c) for c, v in r["pixel"].items())
    (io, co), (inw, cn) = best(old), best(new)
    print(f"\nIoU ดีสุด: WeedyRice only {io:.3f} (conf {co}) | + CoFly {inw:.3f} (conf {cn}) | ต่าง {inw - io:+.3f}")

## 8. ภาพ val ที่โมเดลใหม่พลาดมากที่สุด

เขียว = เจอถูก, แดง = พลาด, น้ำเงิน = ชี้ผิด (เฉพาะ class `weedy_rice`)

In [ ]:
if VAL_JSON.exists():
    import cv2
    from ultralytics import YOLO
    from rwd.evaluate import error_overlay, label_mask, predict_mask, scores
    from rwd.yolo_io import load_data_yaml

    conf = best(new)[1]
    rows = sorted(new["per_image"][conf], key=lambda r: scores(r["tp"], r["fp"], r["fn"])["iou"] or 0)[:3]
    model = YOLO(str(RUN / "weights/best.pt"))
    images_dir = load_data_yaml(cfg["data"]).splits["val"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    for ax, r in zip(axes, rows):
        p = images_dir / r["image"]
        pred = predict_mask(model, p, float(conf), new["imgsz"], cfg.get("device", 0))
        s = scores(r["tp"], r["fp"], r["fn"])
        ax.imshow(error_overlay(cv2.imread(str(p)), pred, label_mask(p, *pred.shape)))
        ax.set_title(f"IoU {s['iou'] or 0:.2f}  recall {s['recall'] or 0:.2f}  {r['image'][13:40]}", fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("ยังไม่มีผล val")

## Next

- ดูสรุปทุกรอบที่ `00_overview.ipynb`
- ถ้า IoU ไม่ดีขึ้นชัดเจน ให้ใช้ `yolo26n-seg` รอบแรกต่อ (ไม่ต้องใช้ CoFly)